# Embedding SigLIP 2 das thumbnails do Gold v2

Rode no Colab com GPU (**Ambiente de execucao > Alterar tipo > T4**) e depois **Executar tudo**.

1. Arraste `emb_v2_bundle.tar.gz` (gerado por `scripts/pipeline/make_embedding_bundle_v2.py`) para o painel **Arquivos**.
2. No fim, baixe `embeddings.npz` e coloque em `data/silver/features/thumbnail_siglip2_v2/<data>/embeddings.npz`.

Mesmo modelo e mesmo calculo do E-08 (`google/siglip2-base-patch16-224`, `get_image_features`, normalizacao L2). As imagens ja vem recortadas e redimensionadas pelo mesmo `preprocess` da v1.

In [ ]:
import os, tarfile, time
import numpy as np, pandas as pd, torch
from PIL import Image
from transformers import AutoModel, AutoProcessor
EMB_MODEL = 'google/siglip2-base-patch16-224'
BUNDLE = '/content/emb_v2_bundle.tar.gz'
assert os.path.exists(BUNDLE), 'arraste emb_v2_bundle.tar.gz para /content'
with tarfile.open(BUNDLE) as tar:
    tar.extractall('/content')
ids = pd.read_csv('/content/emb_v2/ids.csv', dtype=str)['video_id'].tolist()
print(len(ids), 'imagens; GPU:', torch.cuda.is_available())

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModel.from_pretrained(EMB_MODEL).to(device).eval()
processor = AutoProcessor.from_pretrained(EMB_MODEL)
chunks, start_t = [], time.time()
for start in range(0, len(ids), 64):
    batch = [Image.open(f'/content/emb_v2/thumbs/{i}.jpg').convert('RGB') for i in ids[start:start + 64]]
    inputs = processor(images=batch, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.get_image_features(**inputs)
    if not isinstance(out, torch.Tensor):
        out = out.pooler_output if getattr(out, 'pooler_output', None) is not None else out[0]
    chunks.append(torch.nn.functional.normalize(out.float(), dim=-1).cpu().numpy())
    print(f'\r{min(start + 64, len(ids))}/{len(ids)}  {time.time() - start_t:.0f}s', end='')
emb = np.vstack(chunks).astype(np.float32)
print('\n', emb.shape, 'normas ~1:', np.linalg.norm(emb, axis=1)[:3])

In [ ]:
np.savez('/content/embeddings.npz', embeddings=emb, video_id=np.array(ids), source=np.array(EMB_MODEL))
try:
    from google.colab import files
    files.download('/content/embeddings.npz')
except ImportError:
    print('salvo em /content/embeddings.npz')